# Exploratory Data Analysis — NYC Short-Term Rental Prices

Initial exploration of the NYC Airbnb dataset as part of the ML pipeline for short-term rental price estimation.

## 1. Setup & Fetch Data from W&B

Initialise a W&B run and download the `sample.csv` artifact produced by the `download` pipeline step.

In [ ]:
import wandb
import pandas as pd
import os

run = wandb.init(project="nyc_airbnb", group="eda", save_code=True)
artifact_dir = wandb.use_artifact("sample.csv:latest").download(root="./artifacts/sample_csv_latest")
local_path = os.path.join(artifact_dir, "sample1.csv")
df = pd.read_csv(local_path)


## 2. Automated Profiling

Generate a full profile report of the dataset using `ydata_profiling`. Key observations:
- `last_review` is stored as a string and must be converted to datetime.
- `price` contains outliers ($0 and above $350) that will be removed.
- Columns with missing values will be handled in the inference pipeline.

In [ ]:
import ydata_profiling

profile = ydata_profiling.ProfileReport(df)
profile.to_notebook_iframe()

## 3. Data Cleaning

Two corrections based on the profiling report:
1. **Drop `price` outliers** — keep only listings priced between $10 and $350/night.
2. **Convert `last_review` to datetime** — required for correct date-based feature engineering.

> Missing values are not imputed here; they will be handled inside the inference pipeline.

In [ ]:
# Drop outliers
min_price = 10
max_price = 350
idx = df['price'].between(min_price, max_price)
df = df[idx].copy()

# Convert last_review to datetime
df['last_review'] = pd.to_datetime(df['last_review'])

## 4. Verify Cleaned Data

Use `df.info()` to confirm that `price` has no outliers and `last_review` is now `datetime64`.

In [ ]:
df.info()

## 5. Finish W&B Run

Close the W&B run to ensure all logged data and the notebook code are flushed to the server.

In [ ]:
run.finish()